In [ ]:
#montamos nuestro drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install python-docx langdetect stanza transformers

In [ ]:
import docx
from langdetect import detect, DetectorFactory
from glob import glob
import os

In [ ]:
import stanza
stanza.download('en')
# This class tokenize
nlp = stanza.Pipeline(lang='en', processors='tokenize')  

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
import torch
from pathlib import Path
from scipy.special import softmax
import numpy as np
import pandas as pd
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request

In [ ]:
file_path = r"https://www.dropbox.com/s/42ottszmsunrf18/sample_data.csv?dl=1"
df = pd.read_csv( file_path )

### Importing the transformer

In [ ]:
### asdasdsdasdas

In [ ]:
# Tasks:
# emoji, emotion, hate, irony, offensive, sentiment
# stance/abortion, stance/atheism, stance/climate, stance/feminist, stance/hillary

task='offensive'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# download label mapping
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

# PT
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.save_pretrained(MODEL)

### Auxiliar function

In [ ]:
# Preprocess text (username and link placeholders)
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

### Example of Transformer operations

In [ ]:
# Extracting text
text = "Good night 😊"
text = preprocess(text)

# Evaluating
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)

# Getting Scores and Ranking
scores = output[0][0].detach().numpy()
scores = softmax(scores)
ranking = np.argsort(scores)
ranking = ranking[::-1]
for i in range(scores.shape[0]):
    l = labels[ranking[i]]
    s = scores[ranking[i]]
    print(f"{i+1}) {l} {np.round(float(s), 4)}")

### Excuting transformer for all the posts

In [ ]:
df.head()

In [ ]:
 for index, row in df.iterrows():
    
  # Post
  post = row[ "raw_post" ]
  
  doc = docx.Document()
  doc.add_paragraph( row['raw_post'] )
  fullText = []
  for para in doc.paragraphs:
    fullText.append(para.text)
  fullText = ' . '.join(fullText)
  # Get all the sentences in the doc file
  doc = nlp( fullText )

  # this list store all the sentences
  all_sentences_text = []
  for sentence in doc.sentences:
    
    # Get text
    sentence_text = ' '.join([token.text for token in sentence.tokens])
    
    # append sentence
    if not sentence_text.strip() == '' :
      all_sentences_text.append( sentence_text )
  
  #número máximo de palabras por sentencia: 512
  all_sentences_text = [ sentence for sentence in all_sentences_text if len( sentence ) < 512 ]                                                          

  # Defining text
  text = " ".join(all_sentences_text)

  try:
    # Evaluating
    encoded_input = tokenizer(text, return_tensors='pt')
    output = model(**encoded_input)

    # Getting Scores and Ranking
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    ranking = np.argsort(scores)
    ranking = ranking[::-1]
    for i in range(scores.shape[0]):
        l = labels[ranking[i]]
        s = scores[ranking[i]]
        # print(f"{i+1}) {l} {np.round(float(s), 4)}")

        df.loc[ index, l ] = np.round(float(s), 4)
    
  except:
    df.loc[ index, 'offensive' ] = np.nan
    df.loc[ index, 'not-offensive' ] = np.nan

Genereting means of the predictions

In [ ]:
df.index

In [ ]:
df.head()

In [ ]:
# New text

In [ ]:
df1.shape

In [ ]:
df1 = df.dropna().copy()
df1.groupby( ['female'], as_index = False )[['offensive',	'not-offensive']].mean()